<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap6_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explaining a Scoring Model with an LLM: Prompts, a Checker and Fine-Tuning

*Deep Learning in Banking* (Bravo, Maldonado and Óskarsdóttir, Wiley, 2026), Chapter 6: extended case study (2026).

> **Extended (2026).** This lab extends the case study printed in Chapter 6. The case is the same: an XGBoost scoring model on 50,000 mortgage applications, explained with SHAP values and a large language model (LLM), Llama 3.2 3B Instruct, first with prompts and then with fine-tuning (QLoRA). Four things changed. (1) Every answer is now **checked by code**, on 100 test cases, so prompts and models are compared with numbers instead of one example. (2) The fine-tuning data were rewritten: the training answers come from a stronger model, Claude Opus 5.5, and only the answers that pass the checker were kept (the appendix shows how). The old data had problems: every training case was a rejected applicant, and the answers carried another model's chat markers. (3) The lab runs on a free Colab T4 GPU. (4) The XGBoost output is called a **score**, not a probability (Section 1 says why). Each section heading names the algorithm of the chapter that it implements, and the last section lists every change from the book.

**The task.** A bank declines or flags a mortgage application with a scoring model. The analyst, and often the applicant, needs the reason in words. SHAP gives each feature's contribution to this applicant's score. An LLM can turn those numbers into sentences. The risk is that the sentences sound right and say something the numbers do not. This lab measures that risk, and shows three ways to reduce it: a better prompt, examples in the prompt, and fine-tuning.

**What you need.** A Colab GPU (Runtime, Change runtime type, T4 GPU). A Hugging Face account that has accepted the Llama 3.2 licence on the [model page](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct), and a Hugging Face token saved as a Colab secret named `HF_TOKEN` (the key icon on the left). The lab takes about 30 minutes to run.

## 0. Setup (Algorithm 6.1, step 1)

We install the packages that Colab does not have, and we check the GPU. `TRAIN = False` loads a fine-tuned adapter that we trained for you; set it to `True` to train your own in Section 8 (about 20 minutes on a T4).

In [ ]:
!pip install -q -U transformers accelerate datasets peft trl bitsandbytes shap
!pip install -q -U xgboost==3.0.5 fsspec==2025.3.2

In [ ]:
import json, os, re, random, shutil, time, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if "TRAIN" not in globals():
    TRAIN = False          # True: train the adapter yourself in Section 8 (about 20 minutes on a T4)

try:                       # the Hugging Face token, from the Colab secret HF_TOKEN
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("No HF_TOKEN secret found:", e)

print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

DATA = Path(os.environ.get("CH6_DATA", "data_ch6"))
FILES = {"loan_app.csv": "1nrhxfnAkI0bZRXJiWu_JVKusAD9iHBpK",                     # Google Drive file ids
         "ch6_explanation_cases.json": "1mXTTucwUIhqZp6AlBI7_g-DYe1HH-7sT",
         "ch6_explainer_adapter.zip": "1ZDZewNfwzClXIvRwNHrSGHd20UhaESB4"}
DATA.mkdir(exist_ok=True)
for name, file_id in FILES.items():
    if name == "ch6_explainer_adapter.zip" and TRAIN:
        continue           # you will make it yourself
    if not (DATA / name).exists():
        import gdown
        gdown.download(id=file_id, output=str(DATA / name), quiet=True)
print(sorted(p.name for p in DATA.iterdir()))

## 1. The scoring model (Algorithm 6.3, steps 18 to 27)

The data are 50,000 mortgage applications from Freddie Mac's single-family loan-level dataset, with eleven application variables and a target that marks a later delinquency (8.7% of the loans). We keep the model of the book: one-hot encoding of the three categorical variables, standardization of the numerical ones, and XGBoost with `scale_pos_weight` to balance the classes.

**A score, not a probability.** With `scale_pos_weight`, the model weights each delinquent loan about ten times more than a performing one. Its output ranks applicants well, but it is not a calibrated probability of default: its average is far above the 8.7% default rate. That makes it a **scoring model**, which decides who is flagged at a cutoff, here 0.5. A probability-of-default (PD) model is a different object: calibrated to the portfolio, and used to set provisions and capital. So in this lab the explanations talk about the *score*.

**Missing-value codes.** Freddie Mac codes a missing credit score as 9999 and a missing debt-to-income ratio as 999. The book's lab uses these codes as numbers. They are rare here (the next cell counts them), and we keep the book's model so that the cases match. Exercise 1 asks you to fix them.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

df = pd.read_csv(DATA / "loan_app.csv")
print(df.shape, "| delinquency rate", round(df.target.mean(), 4))
print("CREDIT_SCORE == 9999:", int((df.CREDIT_SCORE == 9999).sum()), "| DTI_RATIO == 999:", int((df.DTI_RATIO == 999).sum()))

X = df.drop(columns=["target"]); y = df["target"]
categorical_columns = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
numerical_columns = [c for c in X.columns if c not in categorical_columns]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_enc = pd.DataFrame(encoder.fit_transform(X[categorical_columns]), columns=encoder.get_feature_names_out(categorical_columns))
X = pd.concat([X.drop(columns=categorical_columns), X_enc], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

xgb_model = XGBClassifier(max_depth=3, learning_rate=0.01, n_estimators=200, objective="binary:logistic",
                          eval_metric="logloss", gamma=0.001, subsample=0.632, reg_lambda=0.1,
                          random_state=428, tree_method="hist", n_jobs=-1,
                          scale_pos_weight=np.sum(y_train == 0) / np.sum(y_train == 1))
xgb_model.fit(X_train, y_train)
score_test = xgb_model.predict_proba(X_test)[:, 1]
flag_test = (score_test > 0.5).astype(int)
print("Test AUC:", round(roc_auc_score(y_test, score_test), 3))
print("Mean score:", round(score_test.mean(), 3), "against a delinquency rate of", round(y_test.mean(), 3))
print("Share flagged at 0.5:", round(flag_test.mean(), 3))
print(classification_report(y_test, flag_test, digits=3))

## 2. From a decision to three numbers (Algorithm 6.3, steps 28 to 34)

SHAP values (Lundberg and Lee, 2017) split one applicant's score into contributions of the features. For a tree model, `TreeExplainer` computes them exactly, relative to a background sample of applicants. They are on the model's **log-odds** scale, and they add up, with a base value, to the applicant's score in log-odds. A positive value means that this applicant's value of the feature raised the score **compared with the average applicant**. It is not a general rule about the feature. A credit score of 690 raises the score, because the average applicant has about 749; a credit score of 800 lowers it.

We fix the background sample (100 training applicants) so that your numbers are reproducible.

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_model, X_train.sample(100, random_state=0))
i = int(np.where(flag_test == 1)[0][0])               # the first flagged applicant of the test set
sv = explainer(X_test.iloc[[i]])
print("score", round(float(score_test[i]), 3), "| flagged" if flag_test[i] else "| not flagged")
shap.plots.bar(sv[0], max_display=6)

An LLM sees only what we put in the prompt. For each applicant we keep the decision, the score, and the **three features with the largest absolute SHAP value**, each with its raw value (not the standardized one), its average in the training data, and its SHAP value. We call this a **case**.

The rest of the lab uses a fixed set of cases that comes with the data: 1,100 training cases and 100 test cases, half of them flagged in each set. Our cases were built in the same way, but with SHAP's default background, which draws a random sample. Their SHAP values can differ slightly from the ones you compute, and the fixed set makes your results comparable with ours.

In [ ]:
raw_mean = dict(zip(numerical_columns, scaler.mean_))
share = X_train.drop(columns=numerical_columns).mean().to_dict()
CODES = {"CREDIT_SCORE": 9999, "DTI_RATIO": 999}

def make_case(row, shap_row, flagged, score):
    raw = dict(zip(numerical_columns, scaler.inverse_transform(row[numerical_columns].values.reshape(1, -1))[0]))
    top = sorted(zip(row.index, shap_row.values), key=lambda t: -abs(t[1]))[:3]
    feats = []
    for f, s in top:
        if f in raw:
            v = round(float(raw[f]), 2)
            feats.append({"name": f, "value": v, "mean": round(float(raw_mean[f]), 2), "shap": round(float(s), 4),
                          "code": bool(CODES.get(f) == round(v))})
        else:               # a one-hot feature: 1 if the applicant has this category
            feats.append({"name": f, "value": int(row[f]), "mean": round(float(share[f]), 3), "shap": round(float(s), 4), "code": False})
    return {"pred": int(flagged), "score": round(float(score), 3), "features": feats}

print(json.dumps(make_case(X_test.iloc[i], sv[0], flag_test[i], score_test[i]), indent=1))

cases = json.load(open(DATA / "ch6_explanation_cases.json", encoding="utf-8"))
train_cases, eval_cases = cases["train"], cases["eval"]
print(len(train_cases), "training cases,", len(eval_cases), "test cases;",
      sum(c["pred"] for c in eval_cases), "test cases are flagged")

## 3. The language model (Algorithm 6.1, steps 2 to 5)

We load Llama 3.2 3B Instruct in 4-bit NF4 (the QLoRA format, Dettmers et al., 2023), so that it fits a T4 with room for fine-tuning later. The T4 has no bfloat16 arithmetic, so we compute in float16. Decoding is **greedy** (`do_sample=False`): the same prompt always gives the same answer, which a comparison needs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
BASE = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = "<|finetune_right_pad_id|>"   # a reserved Llama token, not the end-of-turn token
tok.padding_side = "left"
EOT = tok.convert_tokens_to_ids("<|eot_id|>")
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=nf4, device_map="auto",
                                             dtype=torch.float16, attn_implementation="sdpa")
print(f"loaded in {time.time() - t0:.0f} s; GPU memory {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def chat(system, user, shots=()):
    msgs = [{"role": "system", "content": system}]
    for u, a in shots:
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": user})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(m, prompts, batch_size=16, max_new_tokens=450):
    m.eval(); outs, n_tok, t0 = [], 0, time.time()
    for k in range(0, len(prompts), batch_size):
        enc = tok(prompts[k:k + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.inference_mode():
            g = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=[EOT, tok.eos_token_id])
        new = g[:, enc["input_ids"].shape[1]:]
        n_tok += int((new != tok.pad_token_id).sum())
        outs += tok.batch_decode(new, skip_special_tokens=True)
    secs = time.time() - t0
    print(f"{len(prompts)} answers in {secs:.0f} s, {n_tok / secs:.0f} tokens per second")
    return outs, secs, n_tok / secs

## 4. The book's prompt (Algorithms 6.2 and 6.3)

This is the prompt of the book's lab, word for word. Read the answers for three test cases. The line "SHAP Impact" usually gets the sign right, because the prompt gives it. Now read "Effect on Risk" next to it, and the system message: it tells the model that "a higher credit score should always reduce risk". When the SHAP value of an applicant's credit score is positive, the model has to reconcile the two, and often it writes a sentence that contradicts the number. A reader cannot tell which half to trust.

In [ ]:
ORIG_SYSTEM = "\nYou are a financial risk analyst. Your job is to analyze SHAP values and provide structured, fact-based explanations of the model's predictions.\n\nGuidelines:\n1. Interpret SHAP values correctly:\n   - **A positive SHAP value means the feature increases delinquency risk.**\n   - **A negative SHAP value means the feature decreases delinquency risk.**\n2. **Do not contradict basic financial logic**:\n   - A **higher credit score should always reduce risk** unless explicitly stated otherwise.\n3. **Strictly follow the response format. Do not add extra text or repeat information.**\n4. **Do not argue against the given ranking of features.**\n5. **Avoid repetition, unnecessary details, or ranking errors.**\n"
ORIG_USER = "\nThe model predicts that the client is {delinquency_status}.\n\nHere are the three most important features influencing the prediction:\n\n{shap_text}\n\n### Instructions:\n- Analyze how each of three features contributes to the prediction.\n- **Use correct feature names, not feature values.**\n- **Strictly follow the structured response format.**\n- **SHAP values must be interpreted correctly**:\n  - **A positive SHAP value means the feature increases delinquency risk.**\n  - **A negative SHAP value means the feature decreases delinquency risk.**\n\n### Response Format:\nFeature Name: [Feature Name]\nEffect on Risk: Explain whether a higher or lower value increases delinquency risk and why.\nSHAP Impact: Clearly state whether the SHAP value shows an increase or decrease in delinquency risk and explain its significance.\n\n### Example Response:\n\nFeature Name: Credit Score\nEffect on Risk: A higher credit score reduces delinquency risk because it indicates a strong repayment history and financial responsibility.\nSHAP Impact: The SHAP value **-0.5862** shows that **including credit score for this client decreases** the probability of delinquency, meaning the model considers this a strong indicator of financial reliability.\n\nNow begin your structured analysis:\n"

def user_book(c):
    shap_text = "\n".join(f"{f['name']}: SHAP value = {f['shap']:.4f}, feature value = {f['value']:.2f}" for f in c["features"])
    status = "likely to be delinquent" if c["pred"] == 1 else "unlikely to be delinquent"
    return ORIG_USER.replace("{delinquency_status}", status).replace("{shap_text}", shap_text)

out_book, _, _ = generate(model, [chat(ORIG_SYSTEM, user_book(c)) for c in eval_cases[:3]])
for c, o in zip(eval_cases[:3], out_book):
    print("=" * 100); print(json.dumps(c["features"])); print("-" * 100); print(o)

## 5. A checker (beyond the book)

To compare prompts and models we need a measure that does not depend on reading a few answers. We ask for the answer in JSON, a fixed format that code can read, and we check four things for each of the three features:

- **valid**: the answer is JSON with a list of features;
- **coverage**: the feature is named;
- **direction**: it says "raises" for a positive SHAP value and "lowers" for a negative one;
- **numbers**: the value and the SHAP value it quotes are the ones in the case.

An answer **passes** when all four hold for all three features. Passing is the minimum we ask of an answer. One more check reads the reasons: when a reason says that the value is above or below the average (or higher, lower, longer, shorter), we test that against the case. This **comparison** check is the main measure of the lab. It catches the error that sounds most convincing: a reason that explains the direction with a fact that is false.

The checker cannot tell whether a reason is true in the world, or useful to an analyst. Section 11 comes back to this.

In [ ]:
def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

ABOVE = re.compile(r"\b(above|higher than|greater than|more than|exceeds|longer than|larger than)\b", re.I)
BELOW = re.compile(r"\b(below|lower than|less than|shorter than|smaller than|fewer than)\b", re.I)

def check(c, text):
    r = {"valid": 0, "coverage": 0.0, "direction": 0.0, "numbers": 0.0, "pass": 0, "compares": 0, "compares_right": 0}
    a = parse_json(text)
    if not isinstance(a, dict) or not isinstance(a.get("features"), list):
        return r
    r["valid"] = 1
    got = {str(f.get("name", "")).strip(): f for f in a["features"] if isinstance(f, dict)}
    hit = dirn = num = 0
    for f in c["features"]:
        g = got.get(f["name"])
        if g is None:
            continue
        hit += 1
        dirn += str(g.get("direction", "")).lower() == ("raises" if f["shap"] > 0 else "lowers")
        try:
            num += abs(float(g["shap"]) - f["shap"]) < 6e-5 and abs(float(g["value"]) - f["value"]) < 6e-3
        except Exception:
            pass
        up, down = ABOVE.search(str(g.get("reason", ""))), BELOW.search(str(g.get("reason", "")))
        if up or down:
            r["compares"] += 1
            r["compares_right"] += (bool(up) and not down) == (f["value"] > f["mean"])
    k = len(c["features"])
    r.update(coverage=hit / k, direction=dirn / k, numbers=num / k)
    r["pass"] = int(hit == dirn == num == k)
    return r

RESULTS = {}
def score(name, cs, outs, secs=None, tps=None):
    rows = [check(c, o) for c, o in zip(cs, outs)]
    s = {k: round(float(np.mean([r[k] for r in rows])), 3) for k in ("valid", "coverage", "direction", "numbers", "pass")}
    n_cmp = sum(r["compares"] for r in rows)
    s["reasons that compare"] = round(n_cmp / (3 * len(rows)), 3)
    s["comparisons right"] = round(sum(r["compares_right"] for r in rows) / max(n_cmp, 1), 3)
    s["pass, flagged"] = round(float(np.mean([r["pass"] for r, c in zip(rows, cs) if c["pred"] == 1])), 3)
    s["pass, not flagged"] = round(float(np.mean([r["pass"] for r, c in zip(rows, cs) if c["pred"] == 0])), 3)
    if secs is not None:
        s["seconds"], s["tokens per second"] = round(secs), round(tps)
    RESULTS[name] = s
    return pd.Series(s, name=name)

# the checker on the training answers written by Claude Opus 5.5
teacher_outs = [json.dumps(c["answer"]) for c in train_cases]
score("Claude Opus 5.5 (training answers)", train_cases, teacher_outs)

The training answers pass by construction: we kept only the answers that passed (one of 1,100 failed). Their comparisons, which the filter did not check, show what a good answer looks like.

## 6. A structured prompt (Algorithm 6.3, steps 31 to 36, revised)

The book suggests giving the model the original values of the features, not only the SHAP values ("try it!"). The new prompt does that. It gives each feature's raw value with its training average next to the SHAP value, states what a SHAP value means, and asks for JSON. It shows the format in words, without example numbers: in our first version the template showed `"value": 0.0`, and the model copied the zero into many answers. A later version wrote each SHAP value with its sign (`+0.2383`), and in one of our runs a quarter of the answers copied the plus sign into the JSON, where it is not valid. This model copies the form of its input, so the prompt shows numbers in the form we want back.

In [ ]:
SYSTEM = """You explain the output of a credit scoring model to a credit analyst. The model is XGBoost. Its output is a score between 0 and 1 that ranks applicants; it is not a calibrated probability of default. Applicants with a score above 0.5 are flagged.
SHAP values are on the model's log-odds scale. A positive SHAP value means that this feature value raised this applicant's score compared with the average applicant. A negative SHAP value means that it lowered the score. A SHAP value describes this applicant only, compared with the average applicant. It is not a general rule about the feature.
Use only the numbers given. Answer only with JSON in the requested format."""

def user_prompt(c):
    flag = "flagged as likely to be delinquent" if c["pred"] == 1 else "not flagged"
    lines = []
    for k, f in enumerate(c["features"], 1):
        avg = f"training average {f['mean']:.2f}" if f["name"] in numerical_columns else f"share in training data {f['mean']:.3f}"
        note = " (this value is the data provider's code for 'not available')" if f["code"] else ""
        lines.append(f"{k}. {f['name']}: value {f['value']:.2f}{note}, {avg}, SHAP {f['shap']:.4f}")
    return (f"Model decision: {flag} (score {c['score']:.3f}; the cutoff is 0.5).\n"
            "Top three features by absolute SHAP value:\n" + "\n".join(lines) + "\n\n"
            "Return a JSON object with two keys. \"features\" is a list with one object per feature, in the given order, "
            "with the keys \"name\" (the feature name), \"value\" (the value given above), \"shap\" (the SHAP value given above), "
            "\"direction\" (\"raises\" or \"lowers\") and \"reason\" (one sentence). \"summary\" is two sentences for the analyst.")

print(user_prompt(eval_cases[0]))
out_structured, secs, tps = generate(model, [chat(SYSTEM, user_prompt(c)) for c in eval_cases])
print(out_structured[0])
score("Structured prompt", eval_cases, out_structured, secs, tps)

## 7. Two examples in the prompt (beyond the book)

A prompt can also show what a good answer looks like. We add two training cases with Claude's answers, one flagged and one not, as earlier turns of the conversation (few-shot prompting). The prompt gets longer, so each answer costs more.

In [ ]:
ex1 = next(c for c in train_cases if c["pred"] == 1)
ex0 = next(c for c in train_cases if c["pred"] == 0)
shots = [(user_prompt(ex1), json.dumps(ex1["answer"])), (user_prompt(ex0), json.dumps(ex0["answer"]))]
out_shots, secs, tps = generate(model, [chat(SYSTEM, user_prompt(c), shots) for c in eval_cases])
print(out_shots[0])
score("Two examples", eval_cases, out_shots, secs, tps)

## 8. Fine-tuning with QLoRA (Algorithm 6.4)

Now we teach the small model the task with the training answers. This is **distillation**: a large model writes the answers, and a small one learns to write them, so that the bank can run the small one on its own hardware.

**LoRA** (Hu et al., 2022) freezes the model's weights and learns a low-rank update for each linear layer: rank 16 and scale 32 here, about 24 million parameters out of 3.2 billion. **QLoRA** (Dettmers et al., 2023) does this on top of the 4-bit model. Each training example is a prompt (system message and case) and a completion (Claude's answer), and the loss counts only the completion.

The two answers used as examples in Section 7 stay out. Of the other 1,097, the trainer uses 987 for training and 110 to check the loss on answers it did not train on. With `TRAIN = False`, the next cell loads the adapter that we trained with exactly this code (one epoch, 124 steps, 156 s on a slice of an H100 GPU). With `TRAIN = True`, it trains your own. On a T4 the same training took 1,212 s in our pilot, so plan on about 20 minutes.

In [ ]:
from peft import LoraConfig, PeftModel
ADAPTER = DATA / "ch6_explainer_adapter"

if TRAIN:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    rows = [{"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt(c)}],
             "completion": [{"role": "assistant", "content": json.dumps(c["answer"])}]}
            for c in train_cases if c is not ex1 and c is not ex0]
    ds = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=SEED)
    tok.padding_side = "right"
    cfg = SFTConfig(output_dir="ft", per_device_train_batch_size=4, gradient_accumulation_steps=2, num_train_epochs=1,
                    learning_rate=2e-4, fp16=True, bf16=False, logging_steps=10, eval_strategy="steps", eval_steps=40,
                    warmup_steps=10, optim="paged_adamw_8bit", gradient_checkpointing=True, max_length=1024,
                    report_to="none", seed=SEED)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules="all-linear")
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds["train"], eval_dataset=ds["test"],
                         processing_class=tok, peft_config=lora)
    for n, p in trainer.model.named_parameters():   # the adapters come out in bfloat16; the T4's float16 training needs float32
        if p.requires_grad:
            p.data = p.data.float()
    t0 = time.time(); trainer.train(); print(f"trained in {time.time() - t0:.0f} s")
    ft = trainer.model
    ft.save_pretrained(ADAPTER)
    shutil.make_archive(str(DATA / "ch6_explainer_adapter"), "zip", ADAPTER)
    log = pd.DataFrame(trainer.state.log_history)
    log.dropna(subset=["loss"]).plot(x="step", y="loss", title="Training loss"); plt.show()
    del trainer
else:
    with zipfile.ZipFile(DATA / "ch6_explainer_adapter.zip") as z:
        z.extractall(ADAPTER)
    ft = PeftModel.from_pretrained(model, ADAPTER)

n_lora = sum(p.numel() for n, p in ft.named_parameters() if "lora_" in n)
print(f"LoRA parameters: {n_lora / 1e6:.1f} million")
tok.padding_side = "left"
ft.config.use_cache = True
out_ft, secs, tps = generate(ft, [chat(SYSTEM, user_prompt(c)) for c in eval_cases])
print(out_ft[0])
score("Fine-tuned (QLoRA)", eval_cases, out_ft, secs, tps)

## 9. The comparison (Algorithm 6.5, on 100 cases)

In [ ]:
first = ["reasons that compare", "comparisons right", "pass", "pass, flagged", "pass, not flagged"]
table = pd.DataFrame(RESULTS).T
table[first + [k for k in table.columns if k not in first]]

Read the first two columns. The structured prompt compares the applicant with the average in 19% of its reasons. As 45% of those comparisons are right, it looks very close to a random guess (if the classes are balanced). Two examples teach the model to compare (81% of its reasons), but still that leaves an error of close to 20%. The examples show the form of a comparison, but provide limited in-context learning. The fine-tuned model compares in 93% of its reasons and is right in 98% of them, the same accuracy as Claude's training answers, and it compares more often than Claude did (85%). On this task, we match the accuracy of a commercial, Opus-level model with a very small fine-tuned model! This is where I see AI going to in the near future. For some applications, general, very powerful commercial models will be necessary. For well-defined tasks, small, local models deployed either in-house or in private clouds will be the most cost-effective solution.

The pass rate shows a second weakness of the prompts. The structured prompt fails 15 of the 100 cases, all of them flagged. In 14 of them, it says that the third feature lowers the score, when its small positive SHAP value (between 0.03 and 0.05) actually increased it. Two examples cut this to three cases, and the fine-tuned model makes no such error at all. The prompts are also sensitive to small choices of form (Section 6).

## 10. Efficient inference: 16-bit or 4-bit? (beyond the book)

The 4-bit model saves memory. Does it cost quality or speed? We load the base model twice, in float16 and in NF4, and run the structured prompt on 32 test cases (16 flagged and 16 not).

In [ ]:
import gc
del ft, model; gc.collect(); torch.cuda.empty_cache()
sub = [c for c in eval_cases if c["pred"] == 1][:16] + [c for c in eval_cases if c["pred"] == 0][:16]
prompts = [chat(SYSTEM, user_prompt(c)) for c in sub]
for name, kw in [("float16", {"dtype": torch.float16}), ("NF4, 4-bit", {"quantization_config": nf4, "dtype": torch.float16})]:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    m = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto", attn_implementation="sdpa", **kw)
    weights = (torch.cuda.memory_allocated() - before) / 1e9
    outs, secs, tps = generate(m, prompts)
    s = score(f"Base model, {name}", sub, outs, secs, tps)
    RESULTS[f"Base model, {name}"]["weights (GB)"] = round(weights, 2)
    RESULTS[f"Base model, {name}"]["peak memory (GB)"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    del m; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(RESULTS).T.iloc[-2:]

We can see the impact of reducing precision now. In our run, the NF4 weights took 2.2 GB of GPU memory against 6.4 GB in float16, and the peak during generation was 4.4 GB against 8.6 GB. NF4 was slower, at 602 tokens per second against 722, because bitsandbytes unpacks the NF4 weights to 16 bits before each multiplication. Newer GPUs, such as NVIDIA's Blackwell generation, compute directly in other 4-bit formats (FP4). Both versions passed 27 of the 32 cases. The float16 model wrote more comparisons (40% of its reasons against 23%), and both were right about half the time, although with 32 cases we cannot conclude this is a general measure. On the T4 of our pilot (32 flagged cases), the two versions took the same time, and float16 was faster per token (158 against 140 tokens per second). On this task, the 4-bit model saves about half the memory at a small cost in speed and no clear cost in quality.

## 11. What the checker cannot see (beyond the book)

The checker reads the format, the direction, the numbers and the above-or-below comparisons. It does not know whether "a single borrower has less income to absorb a shock" is the reason the model learned, or whether a summary is useful to an analyst. Three ways to go further:

- **Read a sample.** Read twenty answers of each model next to their cases, and mark the reasons that are wrong or empty. A human check of a small sample is still the reference.
- **A model as a judge.** A strong model can grade the reasons against a rubric. Its grades have known biases (position, length, and a preference for its own answers; Zheng et al., 2023), so calibrate the judge on your human marks before you trust it.
- **Stability.** Change the order of the three features, or the wording of the prompt, and check that the answer does not change.

A bank that uses LLM explanations with customers also needs the rules of its supervisor. The explanation must match the reasons the model actually used, and that is what the checker tests.

## 12. Exercises

1. **The missing-value codes.** Replace 9999 in `CREDIT_SCORE` and 999 in `DTI_RATIO` with `np.nan` before the split (XGBoost handles missing values), retrain the model, and find the applicants whose explanation changes. What would the old explanation have told such an applicant about the credit score?
2. **A harder case for the checker.** Write a prompt that asks for a free-text letter to the applicant, in plain language, and change the checker to read the direction words from the letter. Do the prompts and the fine-tuned model keep their order?
3. **Spanish.** Ask the models for the reasons in Spanish. Which checks still work, and which need new word lists?
4. **A tool call.** Write a function `score_and_explain(row)` that returns the case of an applicant, and let the model call it: describe the function in the system prompt, ask the model to answer with a JSON call when it needs an applicant's case, run the function, and return the result as the next turn. This is the pattern of an agent with tools.
5. **Less data.** Train the adapter on 100 and on 300 answers (set `TRAIN = True` and sample `rows`). How many answers does the model need to beat the two-example prompt?

## What is new in this edition (November 2026)

This edition extends the case study of Chapter 6. The five algorithms of the chapter are still here, as the section headings show, but the data for fine-tuning, the prompts and the evaluation are new. The results are not comparable with the single answers printed in the book.

1. **The same scoring model and case.** The XGBoost model has the settings of Algorithm 6.3, and Section 2 explains the same applicant as the book (credit score 683, SHAP value 0.6744). The missing-value codes 9999 and 999 stay as in the book, and Exercise 1 fixes them.
2. **A checker on 100 test cases.** Algorithm 6.5 compares one answer of the fine-tuned model with one of the base model by reading them. This edition scores every prompt and model with code on the same 100 test cases (Section 5).
3. **New training data.** The book's fine-tuning data had three problems: every training case was a rejected applicant, the answers carried another model's chat markers (`<|im_start|>`), and the prompts showed standardized values while the inference prompt showed raw ones. The new data have 1,100 cases from the training split, half of them flagged, with raw values and the same prompt as at inference. Claude Opus 5.5 wrote the answers, and the checker kept the ones that passed (appendix).
4. **The prompts.** The book's prompt is kept word for word in Section 4. Its system message says that "a higher credit score should always reduce risk", which contradicts the SHAP value when an applicant's credit score raised the score. The structured prompt of Section 6 states what a SHAP value means, gives the raw value and the training average, and asks for JSON.
5. **A score, not a probability.** With `scale_pos_weight`, the XGBoost output ranks applicants but is not a calibrated probability of default, so the lab calls it a score (Section 1).
6. **Greedy decoding.** Algorithms 6.2 and 6.5 sample the answer (`top_p` and `temperature`). This edition decodes greedily, so that the same prompt always gives the same answer and the arms can be compared.
7. **A free T4 instead of an L4.** The T4 has no bfloat16 arithmetic, so the model computes in float16. Attention uses PyTorch's `sdpa` instead of `flash_attention_2`, and the pad token is a reserved Llama token instead of the end-of-sequence token (Algorithm 6.1, step 5). Training uses a batch of 4 with 2 accumulation steps instead of 2 with 2, `paged_adamw_8bit` instead of `paged_adamw_32bit`, and casts the adapters to float32 for the float16 gradient scaler.
8. **Prompt and completion instead of one text field.** Algorithm 6.4 (steps 40 to 42) joins the system, user and assistant messages into one `text` field, so the loss also counts the prompt. This edition passes the prompt and Claude's answer separately, and the loss counts only the answer. Weights & Biases (steps 37 and 56) is replaced by a loss plot in the notebook.
9. **A trained adapter.** The notebook loads an adapter trained with the code of Section 8, so the lab fits a Colab session. Training is optional (`TRAIN = True`).
10. **New sections.** Few-shot prompting (Section 7), 16-bit against 4-bit inference (Section 10) and the limits of the checker (Section 11).

## References

- Bravo, C., Maldonado, S. and Óskarsdóttir, M. (2026). *Deep Learning in Banking*. Wiley.
- Chen, T. and Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785
- Dettmers, T., Pagnoni, A., Holtzman, A. and Zettlemoyer, L. (2023). QLoRA: Efficient finetuning of quantized LLMs. *NeurIPS*. arXiv:2305.14314.
- Hu, E. J. et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR*. arXiv:2106.09685.
- Lundberg, S. M. and Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, 30.
- Meta (2024). Llama 3.2 3B Instruct, model card and Llama 3.2 Community License. https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
- Zheng, L. et al. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. *NeurIPS Datasets and Benchmarks*.

## Appendix: how we made the training answers (beyond the book)

This is the code that wrote the training answers, once, in November 2026. It is here so that you can see what the answers are, and do the same for your own task. **Do not run it for this lab**: it needs a paid Anthropic API key, and the answers are already in the data file. `GENERATE = False` keeps it switched off.

We sent the 1,100 training cases, with the system message and the prompt of Section 6 (in an earlier wording that showed the JSON format with zeros and wrote the SHAP values with a plus sign), to Claude Opus 5.5 through Anthropic's **Message Batches API**, which costs half the normal price and returns the answers within 24 hours. The API forced the answers into the JSON schema below. Our batch took 3 hours 34 minutes and cost US$6.04 (953,266 input and 413,840 output tokens). Then the checker of Section 5 filtered the answers: 1,099 passed, and the one that failed had an empty list of features.

In [ ]:
GENERATE = False

SCHEMA = {"type": "object", "additionalProperties": False, "required": ["features", "summary"],
          "properties": {"summary": {"type": "string"},
                         "features": {"type": "array", "items": {"type": "object", "additionalProperties": False,
                                      "required": ["name", "value", "shap", "direction", "reason"],
                                      "properties": {"name": {"type": "string"}, "value": {"type": "number"},
                                                     "shap": {"type": "number"},
                                                     "direction": {"type": "string", "enum": ["raises", "lowers"]},
                                                     "reason": {"type": "string"}}}}}}

if GENERATE:
    !pip install -q anthropic
    import anthropic
    from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
    from anthropic.types.messages.batch_create_params import Request
    client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    requests = [Request(custom_id=f"tr{k}", params=MessageCreateParamsNonStreaming(
                    model="claude-opus-5-5", max_tokens=8000, system=SYSTEM,
                    messages=[{"role": "user", "content": user_prompt(c)}],
                    output_config={"effort": "medium", "format": {"type": "json_schema", "schema": SCHEMA}}))
                for k, c in enumerate(train_cases)]
    batch = client.messages.batches.create(requests=requests)
    print("batch", batch.id, "- keep this id; the results stay available for 29 days")
    while client.messages.batches.retrieve(batch.id).processing_status != "ended":
        time.sleep(60)
    answers = {}
    for r in client.messages.batches.results(batch.id):
        if r.result.type == "succeeded" and r.result.message.stop_reason == "end_turn":
            text = next(b.text for b in r.result.message.content if b.type == "text")
            c = train_cases[int(r.custom_id[2:])]
            if check(c, text)["pass"]:
                answers[r.custom_id] = json.loads(text)
    print(len(answers), "answers pass the checker")